In [1]:
!pip install -q -U langchain langchain-community transformers torch accelerate

In [2]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [12]:
pipe = pipeline(
    task="text-generation",
    model="distilgpt2",
    max_new_tokens=40,
    truncation=True
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [4]:
job_description = """
We are hiring a Data Scientist.

Required skills:
Python, SQL, Machine Learning, Statistics, Data Analysis, Pandas, Scikit-learn

Preferred tools:
Jupyter Notebook, Tableau, Power BI, Git

Experience:
1+ years in data science or analytics

Responsibilities:
Analyze datasets, build predictive models, communicate insights
"""

In [5]:
strong_resume = """
Rahul Sharma
Experience: 3 years as Data Scientist
Skills: Python, SQL, Machine Learning, Pandas, Scikit-learn, Statistics, Data Analysis
Tools: Jupyter, Tableau, Git
"""

average_resume = """
Priya Verma
Experience: 1 year internship
Skills: Python, SQL, Excel
Tools: Jupyter
"""

weak_resume = """
Amit Kumar
Experience: Fresher
Skills: HTML, CSS, JavaScript
Tools: VS Code
"""

In [13]:
extract_prompt = PromptTemplate.from_template("""
Resume:
{resume}

Give answer in exactly 3 lines only:
Skills: ...
Experience: ...
Tools: ...
""")

match_prompt = PromptTemplate.from_template("""
Job Description:
{jd}

Resume:
{resume}

Give answer in exactly 4 lines only:
Matched Skills: ...
Missing Skills: ...
Tool Match: ...
Summary: ...
""")

score_prompt = PromptTemplate.from_template("""
Job Description:
{jd}

Match Result:
{match}

Give answer in exactly 2 lines only:
Fit Score: ...
Reason: ...
""")

explain_prompt = PromptTemplate.from_template("""
Score Result:
{score}

Give answer in exactly 3 short lines only.
""")

In [7]:
parser = StrOutputParser()

extract_chain = extract_prompt | llm | parser
match_chain = match_prompt | llm | parser
score_chain = score_prompt | llm | parser
explain_chain = explain_prompt | llm | parser

In [18]:
import re

def extract_resume_info(resume_text):
    skills_match = re.search(r"Skills:\s*(.*)", resume_text, re.IGNORECASE)
    exp_match = re.search(r"Experience:\s*(.*)", resume_text, re.IGNORECASE)
    tools_match = re.search(r"Tools:\s*(.*)", resume_text, re.IGNORECASE)

    skills = [s.strip() for s in skills_match.group(1).split(",")] if skills_match else []
    experience = exp_match.group(1).strip() if exp_match else "Not mentioned"
    tools = [t.strip() for t in tools_match.group(1).split(",")] if tools_match else []

    return {
        "skills": skills,
        "experience": experience,
        "tools": tools
    }


def extract_jd_info(jd_text):
    req_skills_match = re.search(r"Required skills:\s*(.*?)\n\nPreferred tools:", jd_text, re.IGNORECASE | re.DOTALL)
    tools_match = re.search(r"Preferred tools:\s*(.*?)\n\nExperience:", jd_text, re.IGNORECASE | re.DOTALL)

    req_skills = []
    pref_tools = []

    if req_skills_match:
        req_skills = [s.strip() for s in req_skills_match.group(1).replace("\n", " ").split(",")]

    if tools_match:
        pref_tools = [t.strip() for t in tools_match.group(1).replace("\n", " ").split(",")]

    return {
        "required_skills": req_skills,
        "preferred_tools": pref_tools
    }


def calculate_score(matched_skills, total_skills, matched_tools, total_tools, experience_text):
    skill_score = (len(matched_skills) / total_skills) * 70 if total_skills > 0 else 0
    tool_score = (len(matched_tools) / total_tools) * 20 if total_tools > 0 else 0

    exp_score = 0
    if "3 year" in experience_text.lower():
        exp_score = 10
    elif "1 year" in experience_text.lower():
        exp_score = 7
    elif "intern" in experience_text.lower():
        exp_score = 5
    elif "fresher" in experience_text.lower():
        exp_score = 2

    total_score = round(skill_score + tool_score + exp_score)
    return min(total_score, 100)


def run_pipeline_clean(resume, name):
    jd_info = extract_jd_info(job_description)
    resume_info = extract_resume_info(resume)

    matched_skills = [s for s in resume_info["skills"] if s in jd_info["required_skills"]]
    missing_skills = [s for s in jd_info["required_skills"] if s not in resume_info["skills"]]

    matched_tools = [t for t in resume_info["tools"] if t in jd_info["preferred_tools"]]
    missing_tools = [t for t in jd_info["preferred_tools"] if t not in resume_info["tools"]]

    score = calculate_score(
        matched_skills,
        len(jd_info["required_skills"]),
        matched_tools,
        len(jd_info["preferred_tools"]),
        resume_info["experience"]
    )

    if score >= 75:
        summary = "Strong match for the role."
    elif score >= 45:
        summary = "Average match for the role."
    else:
        summary = "Weak match for the role."

    print("=" * 60)
    print(f"RUNNING FOR: {name}")
    print("=" * 60)

    print("\n[1] Extracted Info:")
    print("Skills:", ", ".join(resume_info["skills"]))
    print("Experience:", resume_info["experience"])
    print("Tools:", ", ".join(resume_info["tools"]))

    print("\n[2] Match Result:")
    print("Matched Skills:", ", ".join(matched_skills) if matched_skills else "None")
    print("Missing Skills:", ", ".join(missing_skills) if missing_skills else "None")
    print("Matched Tools:", ", ".join(matched_tools) if matched_tools else "None")
    print("Missing Tools:", ", ".join(missing_tools) if missing_tools else "None")
    print("Summary:", summary)

    print("\n[3] Score Result:")
    print("Fit Score:", score, "/100")

    print("\n[4] Explanation:")
    print(f"This candidate matches {len(matched_skills)} required skills and {len(matched_tools)} preferred tools.")
    print(f"Experience level mentioned: {resume_info['experience']}.")
    print(summary)

In [22]:
run_pipeline_clean(strong_resume, "Strong Candidate")

RUNNING FOR: Strong Candidate

[1] Extracted Info:
Skills: Python, SQL, Machine Learning, Pandas, Scikit-learn, Statistics, Data Analysis
Experience: 3 years as Data Scientist
Tools: Jupyter, Tableau, Git

[2] Match Result:
Matched Skills: Python, SQL, Machine Learning, Pandas, Scikit-learn, Statistics, Data Analysis
Missing Skills: None
Matched Tools: Tableau, Git
Missing Tools: Jupyter Notebook, Power BI
Summary: Strong match for the role.

[3] Score Result:
Fit Score: 90 /100

[4] Explanation:
This candidate matches 7 required skills and 2 preferred tools.
Experience level mentioned: 3 years as Data Scientist.
Strong match for the role.


In [23]:
run_pipeline_clean(average_resume, "Average Candidate")

RUNNING FOR: Average Candidate

[1] Extracted Info:
Skills: Python, SQL, Excel
Experience: 1 year internship
Tools: Jupyter

[2] Match Result:
Matched Skills: Python, SQL
Missing Skills: Machine Learning, Statistics, Data Analysis, Pandas, Scikit-learn
Matched Tools: None
Missing Tools: Jupyter Notebook, Tableau, Power BI, Git
Summary: Weak match for the role.

[3] Score Result:
Fit Score: 27 /100

[4] Explanation:
This candidate matches 2 required skills and 0 preferred tools.
Experience level mentioned: 1 year internship.
Weak match for the role.


In [24]:
run_pipeline_clean(weak_resume, "Weak Candidate")

RUNNING FOR: Weak Candidate

[1] Extracted Info:
Skills: HTML, CSS, JavaScript
Experience: Fresher
Tools: VS Code

[2] Match Result:
Matched Skills: None
Missing Skills: Python, SQL, Machine Learning, Statistics, Data Analysis, Pandas, Scikit-learn
Matched Tools: None
Missing Tools: Jupyter Notebook, Tableau, Power BI, Git
Summary: Weak match for the role.

[3] Score Result:
Fit Score: 2 /100

[4] Explanation:
This candidate matches 0 required skills and 0 preferred tools.
Experience level mentioned: Fresher.
Weak match for the role.
